# Modelo2 v3: PPG -> ECG (lead II) + clasificación multi-etiqueta de patologías

Modelo multi-tarea (BIDMC, PPG real):
1. Reconstrucción de ECG (regresión de la forma de onda).
2. Clasificación multi-etiqueta de 4 patologías (bradicardia, taquicardia, arritmia, hipoxemia).

Cambios respecto a v2:
- Encoder no causal (conv + BiGRU): el PPG retrasa al ECG, el modelo necesita ver el futuro.
- Decoder aprendido (ConvTranspose) y grafo a T/down pasos (más rápido).
- Entrada de 3 canales (PPG, dPPG, ddPPG): se elimina la fuga del canal SpO2 hacia la etiqueta de hipoxemia.
- pos_weight conectado a la BCE, ReduceLROnPlateau en lugar de OneCycle, checkpoints versionados por configuración.
- Test sin solapamiento (stride = T) para el reporte formal.
- Barras de progreso en carga de datos, pos_weight, entrenamiento, validación, test y diagnóstico.

In [ ]:
# %pip install numpy torch scipy wfdb scikit-learn tqdm ipywidgets

In [ ]:
import os
import math
import json
import random
import argparse
import datetime
import platform
import time
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import wfdb
from tqdm.auto import tqdm
from scipy.signal import correlate, correlation_lags
from scipy.stats import pearsonr
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix

DEFAULT_BIDMC = "bidmc-ppg-and-respiration-dataset-1.0.0"
PATHOLOGY_NAMES = ["bradicardia", "taquicardia", "arritmia", "hipoxemia"]

<h2>Utilidades</h2>

In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def to_native(obj):
    if isinstance(obj, dict):
        return {k: to_native(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_native(v) for v in obj]
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

<h2>Datasets</h2>

In [ ]:
class BIDMCDataset(Dataset):
    """
    PPG real (BIDMC) con ventanas solapadas por registro. Un registro = un paciente,
    por lo que el split por registro es un split por paciente.

    - Salida de ECG: lead II, forma [T, 1].
    - Entrada: [PPG, dPPG, ddPPG], forma [T, 3] (sin SpO2 para evitar fuga hacia la etiqueta).
    - z-score una sola vez por registro.
    - Etiquetas: HR y SpO2 numéricos de BIDMC (archivos *n). Arritmia no tiene anotación aquí,
      su valid_mask es siempre False.
    """
    def __init__(self, root_dir: str, split: str = "train", T: int = 500,
                 stride: int = 250, seed: int = 42):
        self.T = T
        self.split = split
        self.fs = 125

        path = Path(root_dir)
        all_records = sorted({f.stem for f in path.glob("bidmc[0-9][0-9].hea")})
        if not all_records:
            raise FileNotFoundError(f"No se encontraron registros bidmcNN.hea en {path}")

        rng = random.Random(seed)
        rng.shuffle(all_records)
        n = len(all_records)
        if split == "train":
            recs = all_records[:int(n * 0.70)]
        elif split == "val":
            recs = all_records[int(n * 0.70): int(n * 0.85)]
        else:
            recs = all_records[int(n * 0.85):]

        self.record_data: Dict[str, Dict] = {}
        self.windows: List[Tuple[str, int]] = []
        n_skipped = 0

        for rec in tqdm(recs, desc=f"[BIDMC {split}] cargando registros", unit="rec", leave=False):
            rec_path = str(path / rec)
            data = self._load_record(rec_path)
            if data is None:
                n_skipped += 1
                continue
            self.record_data[rec_path] = data
            total = len(data["ppg"])
            if total < T:
                self.windows.append((rec_path, 0))
            else:
                for start in range(0, total - T + 1, stride):
                    self.windows.append((rec_path, start))

        print(f"[BIDMC {split:5s}] {len(self.record_data)} registros válidos "
              f"({n_skipped} descartados) -> {len(self.windows)} ventanas (T={T}, stride={stride})")

    @staticmethod
    def _load_record(rec_path: str) -> Optional[Dict]:
        try:
            record = wfdb.rdrecord(rec_path)
            sig = record.p_signal.astype(np.float32)
            names = [n.lower().strip().rstrip(",") for n in record.sig_name]
            numerics = wfdb.rdrecord(rec_path + "n")
            num_names = [n.lower().strip().rstrip(",") for n in numerics.sig_name]
            if "spo2" not in num_names or "hr" not in num_names:
                return None
            spo2_full = numerics.p_signal[:, num_names.index("spo2")]
            hr_full = numerics.p_signal[:, num_names.index("hr")]
            ppg_idx = next(i for i, n in enumerate(names) if "pleth" in n)
            ecg_idx = next(i for i, n in enumerate(names) if n == "ii")
        except Exception as e:
            print(f"[BIDMC] descartando {rec_path} ({type(e).__name__}: {e})")
            return None

        ppg_raw, ecg_raw = sig[:, ppg_idx], sig[:, ecg_idx]
        return {
            "ppg": ((ppg_raw - ppg_raw.mean()) / (ppg_raw.std() + 1e-8)).astype(np.float32),
            "ecg": ((ecg_raw - ecg_raw.mean()) / (ecg_raw.std() + 1e-8)).astype(np.float32),
            "spo2_full": spo2_full,
            "hr_full": hr_full,
        }

    def __len__(self):
        return len(self.windows)

    def _crop(self, arr: np.ndarray, start: int) -> np.ndarray:
        seg = arr[start:start + self.T]
        if len(seg) < self.T:
            seg = np.concatenate([seg, np.zeros(self.T - len(seg), dtype=np.float32)])
        return seg

    def __getitem__(self, idx):
        rec_path, start = self.windows[idx]
        data = self.record_data[rec_path]

        ppg = torch.tensor(self._crop(data["ppg"], start), dtype=torch.float32)
        ecg = torch.tensor(self._crop(data["ecg"], start), dtype=torch.float32).unsqueeze(-1)  # [T, 1]
        dppg = torch.zeros_like(ppg); dppg[1:] = ppg[1:] - ppg[:-1]
        ddppg = torch.zeros_like(ppg); ddppg[1:] = dppg[1:] - dppg[:-1]
        x = torch.stack([ppg, dppg, ddppg], dim=-1)  # [T, 3]

        t0_s, t1_s = start // self.fs, (start + self.T) // self.fs + 1
        spo2_win = data["spo2_full"][t0_s:t1_s]; spo2_win = spo2_win[~np.isnan(spo2_win)]
        hr_win = data["hr_full"][t0_s:t1_s]; hr_win = hr_win[~np.isnan(hr_win)]
        spo2_mean = float(np.mean(spo2_win)) if len(spo2_win) else float("nan")
        hr_mean = float(np.mean(hr_win)) if len(hr_win) else float("nan")

        y = torch.zeros(4)
        valid = torch.zeros(4, dtype=torch.bool)
        if not math.isnan(hr_mean):
            y[0] = 1.0 if hr_mean < 60 else 0.0
            y[1] = 1.0 if hr_mean > 100 else 0.0
            valid[0] = valid[1] = True
        if not math.isnan(spo2_mean):
            y[3] = 1.0 if spo2_mean < 95.0 else 0.0
            valid[3] = True
        return x, ecg, y, valid


def ppg_ecg_lag(ds: BIDMCDataset, max_lag_s: float = 0.6, secs: int = 120) -> Dict[str, float]:
    """
    Diagnóstico: retraso estimado del PPG respecto al ECG por registro (segundos).
    >0 significa que el PPG llega después del ECG. Es indicativo: con HR alta puede
    confundirse con el ciclo siguiente.
    """
    fs, out = ds.fs, {}
    for rec, d in tqdm(ds.record_data.items(), desc="lag PPG-ECG", unit="rec", leave=False):
        p = d["ppg"][:fs * secs]
        e = np.convolve(np.abs(np.diff(d["ecg"][:fs * secs], prepend=0)), np.ones(9) / 9, "same")
        p = (p - p.mean()) / (p.std() + 1e-8)
        e = (e - e.mean()) / (e.std() + 1e-8)
        c = correlate(p, e, mode="full")
        lags = correlation_lags(len(p), len(e), mode="full")
        m = np.abs(lags) <= int(max_lag_s * fs)
        out[Path(rec).name] = round(float(lags[m][np.argmax(c[m])]) / fs, 3)
    return out

<h2>Grafo cardíaco y modelo</h2>

In [ ]:
def build_mlp(in_dim, hidden_dims, out_dim, activation=nn.SiLU, dropout=0.0):
    layers, prev = [], in_dim
    for h in hidden_dims:
        layers += [nn.Linear(prev, h), activation()]
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        prev = h
    layers.append(nn.Linear(prev, out_dim))
    return nn.Sequential(*layers)


def scatter_sum(src, index, dim_size):
    B, E, H = src.shape
    out = src.new_zeros(B, dim_size, H)
    out.index_add_(1, index, src)
    return out


@dataclass
class HeartGraphSpec:
    node_names: List[str]
    node_types: List[int]
    chambers: List[int]
    coords: torch.Tensor
    edge_index: torch.Tensor
    edge_attr: torch.Tensor


def build_default_heart_graph(device="cpu") -> HeartGraphSpec:
    node_names = [
        "SA", "AV", "HIS", "LBB", "RBB",
        "LP1", "LP2", "LP3", "RP1", "RP2",
        "LA_ant", "LA_post", "RA_ant", "RA_post",
        "LV_septal", "LV_anterior", "LV_lateral", "LV_posterior", "LV_apex", "LV_base",
        "RV_septal", "RV_freewall", "RV_apex", "RV_base",
    ]
    type_map = {
        "SA": 0, "AV": 1, "HIS": 2, "LBB": 2, "RBB": 2,
        "LP1": 3, "LP2": 3, "LP3": 3, "RP1": 3, "RP2": 3,
        "LA_ant": 4, "LA_post": 4, "RA_ant": 4, "RA_post": 4,
        "LV_septal": 5, "LV_anterior": 5, "LV_lateral": 5, "LV_posterior": 5, "LV_apex": 5, "LV_base": 5,
        "RV_septal": 5, "RV_freewall": 5, "RV_apex": 5, "RV_base": 5,
    }
    chamber_map = {
        "SA": 0, "AV": 0, "HIS": 0, "LBB": 5, "RBB": 5,
        "LP1": 5, "LP2": 5, "LP3": 5, "RP1": 5, "RP2": 5,
        "LA_ant": 1, "LA_post": 1, "RA_ant": 2, "RA_post": 2,
        "LV_septal": 3, "LV_anterior": 3, "LV_lateral": 3, "LV_posterior": 3, "LV_apex": 3, "LV_base": 3,
        "RV_septal": 4, "RV_freewall": 4, "RV_apex": 4, "RV_base": 4,
    }
    coords_dict = {
        "SA": (-0.2, 0.7, 0.2), "AV": (-0.1, 0.3, 0.0), "HIS": (0.0, 0.1, 0.0),
        "LBB": (-0.2, -0.1, -0.1), "RBB": (0.2, -0.1, -0.1),
        "LP1": (-0.35, -0.35, -0.1), "LP2": (-0.45, -0.45, -0.2), "LP3": (-0.25, -0.55, -0.25),
        "RP1": (0.35, -0.35, -0.1), "RP2": (0.25, -0.55, -0.2),
        "LA_ant": (-0.15, 0.55, 0.1), "LA_post": (-0.10, 0.45, -0.1),
        "RA_ant": (0.15, 0.55, 0.1), "RA_post": (0.10, 0.45, -0.1),
        "LV_septal": (-0.10, -0.30, 0.0), "LV_anterior": (-0.20, -0.35, 0.15),
        "LV_lateral": (-0.45, -0.35, 0.0), "LV_posterior": (-0.25, -0.40, -0.2),
        "LV_apex": (-0.20, -0.65, -0.15), "LV_base": (-0.15, -0.10, 0.0),
        "RV_septal": (0.08, -0.28, 0.0), "RV_freewall": (0.32, -0.35, 0.0),
        "RV_apex": (0.20, -0.58, -0.12), "RV_base": (0.12, -0.12, 0.0),
    }
    idx = {n: i for i, n in enumerate(node_names)}

    def e(a, b, delay, weight, kind):
        return (idx[a], idx[b], [delay, weight, kind])

    edges = [
        e("SA", "RA_ant", 0.020, 1.0, 0), e("SA", "LA_ant", 0.030, 0.9, 0),
        e("SA", "AV", 0.050, 1.0, 0), e("AV", "HIS", 0.080, 1.0, 0),
        e("HIS", "LBB", 0.015, 1.0, 0), e("HIS", "RBB", 0.015, 1.0, 0),
        e("LBB", "LP1", 0.010, 1.0, 0), e("LBB", "LP2", 0.012, 0.9, 0), e("LBB", "LP3", 0.014, 0.9, 0),
        e("RBB", "RP1", 0.010, 1.0, 0), e("RBB", "RP2", 0.013, 0.9, 0),
    ]
    lv = ["LV_septal", "LV_anterior", "LV_lateral", "LV_posterior", "LV_apex", "LV_base"]
    rv = ["RV_septal", "RV_freewall", "RV_apex", "RV_base"]
    for n in lv:
        edges += [e("LP1", n, 0.010, 0.8, 0), e("LP2", n, 0.012, 0.7, 0)]
    for n in rv:
        edges += [e("RP1", n, 0.010, 0.8, 0), e("RP2", n, 0.012, 0.7, 0)]
    for n in ["LA_ant", "LA_post"]:
        edges += [e("LA_ant", n, 0.020, 0.7, 1), e(n, "LA_ant", 0.020, 0.7, 1)]
    for n in ["RA_ant", "RA_post"]:
        edges += [e("RA_ant", n, 0.020, 0.7, 1), e(n, "RA_ant", 0.020, 0.7, 1)]
    for a, b in zip(lv, lv[1:]):
        edges += [e(a, b, 0.025, 0.7, 1), e(b, a, 0.025, 0.7, 1)]
    for a, b in zip(rv, rv[1:]):
        edges += [e(a, b, 0.040, 0.5, 2), e(b, a, 0.040, 0.5, 2)]

    src = torch.tensor([x[0] for x in edges], dtype=torch.long, device=device)
    dst = torch.tensor([x[1] for x in edges], dtype=torch.long, device=device)
    return HeartGraphSpec(
        node_names=node_names,
        node_types=[type_map[n] for n in node_names],
        chambers=[chamber_map[n] for n in node_names],
        coords=torch.tensor([coords_dict[n] for n in node_names], dtype=torch.float32, device=device),
        edge_index=torch.stack([src, dst], dim=0),
        edge_attr=torch.tensor([x[2] for x in edges], dtype=torch.float32, device=device),
    )


class GraphGRUCell(nn.Module):
    def __init__(self, hidden_dim, edge_dim, msg_dim, dropout=0.0):
        super().__init__()
        self.msg_net = build_mlp(2 * hidden_dim + edge_dim, [msg_dim], msg_dim, dropout=dropout)
        self.gru = nn.GRUCell(msg_dim + hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, h, local_drive, edge_index, edge_attr):
        src_idx, dst_idx = edge_index
        B, N, H = h.shape
        ea = edge_attr.unsqueeze(0).expand(B, -1, -1)
        msgs = self.msg_net(torch.cat([h[:, src_idx], h[:, dst_idx], ea], dim=-1))
        agg = scatter_sum(msgs, dst_idx, dim_size=N)
        inp = torch.cat([agg, local_drive], dim=-1)
        h_new = self.gru(inp.reshape(B * N, -1), h.reshape(B * N, H)).reshape(B, N, H)
        return self.norm(h_new)


class ECGProjectionHead(nn.Module):
    def __init__(self, hidden_dim, n_leads, source_dim=3):
        super().__init__()
        self.node_to_lead = build_mlp(hidden_dim + source_dim, [hidden_dim], n_leads)
        self.source_head = build_mlp(hidden_dim, [hidden_dim // 2], source_dim)

    def forward(self, h, coords):
        B = h.shape[0]
        feat = torch.cat([h, coords.unsqueeze(0).expand(B, -1, -1)], dim=-1)
        return self.node_to_lead(feat).mean(dim=1), self.source_head(h).mean(dim=1)


class OximetryEncoder(nn.Module):
    """No causal: conv + BiGRU. Reduce T a T/down y ve toda la ventana."""
    def __init__(self, input_dim, hidden_dim, down=2):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(input_dim, hidden_dim // 2, 7, padding=3),
            nn.BatchNorm1d(hidden_dim // 2), nn.SiLU(),
            nn.Conv1d(hidden_dim // 2, hidden_dim, 2 * down, stride=down, padding=down // 2),
            nn.BatchNorm1d(hidden_dim), nn.SiLU(),
        )
        self.bigru = nn.GRU(hidden_dim, hidden_dim // 2, batch_first=True, bidirectional=True)

    def forward(self, x):                                   # [B, T, C]
        z = self.conv(x.transpose(1, 2)).transpose(1, 2)    # [B, T/down, H]
        return self.bigru(z)[0]


class PathologyHead(nn.Module):
    def __init__(self, hidden_dim, n_pathologies=4, dropout=0.1):
        super().__init__()
        self.mlp = build_mlp(hidden_dim, [hidden_dim, hidden_dim // 2], n_pathologies, dropout=dropout)

    def forward(self, h_seq):                               # [B, Tp, N, H]
        return self.mlp(h_seq.mean(dim=(1, 2)))


class BioMEMOximetryToECG(nn.Module):
    def __init__(self, graph, input_dim=3, hidden_dim=64, node_emb_dim=32,
                 type_vocab=6, chamber_vocab=6, edge_dim=3, msg_dim=128,
                 n_leads=1, n_layers=1, dropout=0.15, down=2):
        super().__init__()
        self.n_nodes = len(graph.node_names)
        self.down = down

        self.type_emb = nn.Embedding(type_vocab, node_emb_dim)
        self.chamber_emb = nn.Embedding(chamber_vocab, node_emb_dim)
        self.coord_proj = nn.Linear(3, node_emb_dim)
        self.node_init = build_mlp(3 * node_emb_dim, [hidden_dim, hidden_dim], hidden_dim, dropout=dropout)

        self.oximetry_encoder = OximetryEncoder(input_dim, hidden_dim, down=down)
        self.obs_to_nodes = build_mlp(2 * hidden_dim, [hidden_dim], hidden_dim, dropout=dropout)
        self.obs_gate = build_mlp(2 * hidden_dim, [hidden_dim], hidden_dim, dropout=dropout)
        self.cells = nn.ModuleList([
            GraphGRUCell(hidden_dim, edge_dim, msg_dim, dropout=dropout) for _ in range(n_layers)
        ])

        self.ion_head = build_mlp(hidden_dim, [hidden_dim], 1)
        self.tension_head = build_mlp(hidden_dim, [hidden_dim], 1)
        self.phase_head = build_mlp(hidden_dim, [hidden_dim], 2)
        self.ecg_head = ECGProjectionHead(hidden_dim, n_leads, source_dim=3)

        self.temporal_refine = nn.GRU(n_leads + 3, hidden_dim, batch_first=True, bidirectional=True)
        self.out_head = build_mlp(2 * hidden_dim, [hidden_dim], n_leads)
        self.up = nn.ConvTranspose1d(2 * hidden_dim, hidden_dim, 2 * down, stride=down, padding=down // 2)
        self.dec = nn.Sequential(
            nn.SiLU(), nn.Conv1d(hidden_dim, hidden_dim // 2, 5, padding=2),
            nn.SiLU(), nn.Conv1d(hidden_dim // 2, n_leads, 5, padding=2),
        )
        self.pathology_head = PathologyHead(hidden_dim, n_pathologies=4, dropout=dropout)

        self.register_buffer("node_types", torch.tensor(graph.node_types, dtype=torch.long))
        self.register_buffer("chambers", torch.tensor(graph.chambers, dtype=torch.long))
        self.register_buffer("coords", graph.coords.clone())
        self.register_buffer("edge_index", graph.edge_index.clone())
        self.register_buffer("edge_attr", graph.edge_attr.clone())

    def initial_node_state(self, batch_size):
        feats = torch.cat([self.type_emb(self.node_types), self.chamber_emb(self.chambers),
                           self.coord_proj(self.coords)], dim=-1)
        return self.node_init(feats).unsqueeze(0).expand(batch_size, -1, -1).contiguous()

    def step(self, h, obs_t):
        obs_node = obs_t.unsqueeze(1).expand(-1, self.n_nodes, -1)
        fused = torch.cat([obs_node, h], dim=-1)
        local_drive = torch.sigmoid(self.obs_gate(fused)) * self.obs_to_nodes(fused)
        for cell in self.cells:
            h = cell(h, local_drive, self.edge_index, self.edge_attr)
        ecg_t, source_t = self.ecg_head(h, self.coords)
        aux = {"vm": self.ion_head(h).squeeze(-1), "tension": self.tension_head(h).squeeze(-1),
               "phase": self.phase_head(h), "source": source_t}
        return h, ecg_t, aux

    def forward(self, x, h0=None, return_aux=True):
        B, T, _ = x.shape
        assert T % self.down == 0, f"T={T} debe ser múltiplo de down={self.down}"
        h = self.initial_node_state(B) if h0 is None else h0
        obs_seq = self.oximetry_encoder(x)                  # [B, Tp, H]
        Tp = obs_seq.shape[1]

        ecg_raw_seq, source_seq, vm_seq, tension_seq, phase_seq, node_states = [], [], [], [], [], []
        for t in range(Tp):
            h, ecg_t, aux = self.step(h, obs_seq[:, t, :])
            ecg_raw_seq.append(ecg_t); source_seq.append(aux["source"])
            vm_seq.append(aux["vm"]); tension_seq.append(aux["tension"])
            phase_seq.append(aux["phase"]); node_states.append(h)

        ecg_raw = torch.stack(ecg_raw_seq, dim=1)           # [B, Tp, L]
        source = torch.stack(source_seq, dim=1)             # [B, Tp, 3]
        z, _ = self.temporal_refine(torch.cat([ecg_raw, source], dim=-1))   # [B, Tp, 2H]
        pathology_logits = self.pathology_head(torch.stack(node_states, dim=1))

        coarse = (self.out_head(z) + ecg_raw).transpose(1, 2)               # [B, L, Tp]
        skip = F.interpolate(coarse, size=T, mode="linear", align_corners=False)
        fine = self.dec(self.up(z.transpose(1, 2)))                         # [B, L, T]
        ecg = (skip + fine).transpose(1, 2)                                 # [B, T, L]

        out = {"ecg": ecg, "pathology_logits": pathology_logits}
        if return_aux:
            out.update({"source": source, "vm": torch.stack(vm_seq, dim=1),
                        "tension": torch.stack(tension_seq, dim=1),
                        "phase": torch.stack(phase_seq, dim=1), "final_state": h})
        return out

<h2>Función de pérdida</h2>

In [ ]:
def masked_bce(logits, targets, valid_mask, pos_weight=None):
    loss = F.binary_cross_entropy_with_logits(logits, targets, reduction="none", pos_weight=pos_weight)
    loss = loss * valid_mask.float()
    return loss.sum() / valid_mask.float().sum().clamp(min=1.0)


class ImprovedLoss(nn.Module):
    """MSE + Pearson + suavidad + regularización fisiológica + BCE enmascarada (patologías)."""
    def __init__(self, mse_w=1.0, pearson_w=0.3, smooth_w=0.01, phys_w=0.05,
                 source_w=0.01, pathology_w=0.2, pos_weight=None):
        super().__init__()
        self.mse_w, self.pearson_w, self.smooth_w = mse_w, pearson_w, smooth_w
        self.phys_w, self.source_w, self.pathology_w = phys_w, source_w, pathology_w
        self.register_buffer("pos_weight", pos_weight)

    @staticmethod
    def pearson_loss(pred, target):
        vp = pred - pred.mean(dim=1, keepdim=True)
        vt = target - target.mean(dim=1, keepdim=True)
        corr = (vp * vt).sum(dim=1) / (vp.norm(dim=1) * vt.norm(dim=1) + 1e-8)
        return (1.0 - corr).mean()

    @staticmethod
    def physiological_regularization(pred):
        reg = 0.1 * (pred["vm"] ** 2).mean()
        reg = reg + 0.1 * F.relu(pred["tension"].abs() - 5.0).mean()
        dsrc = pred["source"][:, 1:, :] - pred["source"][:, :-1, :]
        return reg + 0.2 * (dsrc ** 2).mean()

    def forward(self, pred, target_ecg, target_pathology=None, valid_mask=None):
        ecg = pred["ecg"]
        l_mse = F.mse_loss(ecg, target_ecg)
        l_pearson = self.pearson_loss(ecg, target_ecg)
        l_smooth = ((ecg[:, 1:] - ecg[:, :-1]) ** 2).mean()
        l_phys = self.physiological_regularization(pred)
        l_source = (pred["source"] ** 2).mean()

        total = (self.mse_w * l_mse + self.pearson_w * l_pearson + self.smooth_w * l_smooth
                 + self.phys_w * l_phys + self.source_w * l_source)
        losses = {"loss": total, "mse": l_mse, "pearson": l_pearson, "smooth": l_smooth,
                  "phys": l_phys, "source_reg": l_source}

        if target_pathology is not None:
            if valid_mask is None:
                valid_mask = torch.ones_like(target_pathology, dtype=torch.bool)
            l_patho = masked_bce(pred["pathology_logits"], target_pathology, valid_mask, self.pos_weight)
            losses["loss"] = losses["loss"] + self.pathology_w * l_patho
            losses["pathology_bce"] = l_patho
        return losses


def compute_pathology_pos_weight(dataset, cap: float = 6.0) -> torch.Tensor:
    ys, ms = [], []
    for i in tqdm(range(len(dataset)), desc="Calculando pos_weight", unit="ventana", leave=False):
        _, _, y, m = dataset[i]
        ys.append(y); ms.append(m)
    Y, M = torch.stack(ys), torch.stack(ms).float()
    pos = (Y * M).sum(dim=0)
    neg = ((1 - Y) * M).sum(dim=0)
    pw = (neg / pos.clamp(min=1)).clamp(max=cap)
    print("pos_weight (train):", {n: round(v, 2) for n, v in zip(PATHOLOGY_NAMES, pw.tolist())})
    return pw

<h2>Métricas</h2>

In [ ]:
def compute_metrics(pred_ecg: torch.Tensor, true_ecg: torch.Tensor) -> Dict:
    pred, true = pred_ecg.detach().cpu().numpy(), true_ecg.detach().cpu().numpy()
    rmse_l, prd_l, corr_l = [], [], []
    for lead in range(pred.shape[-1]):
        p, t = pred[:, :, lead].ravel(), true[:, :, lead].ravel()
        rmse_l.append(float(np.sqrt(np.mean((p - t) ** 2))))
        prd_l.append(float(100.0 * np.sqrt(np.sum((p - t) ** 2) / (np.sum(t ** 2) + 1e-8))))
        corr_l.append(float(pearsonr(p, t)[0]) if p.std() > 1e-6 and t.std() > 1e-6 else 0.0)
    return {"rmse_mean": float(np.mean(rmse_l)), "prd_mean": float(np.mean(prd_l)),
            "corr_mean": float(np.mean(corr_l)), "prd_per_lead": prd_l, "corr_per_lead": corr_l}


def compute_pathology_metrics(logits, targets, valid_mask, threshold=0.5) -> Dict:
    """Solo evalúa filas válidas por clase; las clases sin muestras válidas se excluyen del macro F1."""
    preds = (torch.sigmoid(logits).detach().cpu().numpy() >= threshold).astype(int)
    true = targets.detach().cpu().numpy().astype(int)
    mask = valid_mask.detach().cpu().numpy().astype(bool)

    per_class, f1s = {}, []
    for i, name in enumerate(PATHOLOGY_NAMES):
        col = mask[:, i]
        if col.sum() == 0:
            per_class[name] = {"f1": None, "precision": None, "recall": None, "support": 0,
                               "n_valid_samples": 0, "confusion_matrix": None}
            continue
        yt, yp = true[col, i], preds[col, i]
        f1 = f1_score(yt, yp, zero_division=0)
        per_class[name] = {
            "f1": round(float(f1), 4),
            "precision": round(float(precision_score(yt, yp, zero_division=0)), 4),
            "recall": round(float(recall_score(yt, yp, zero_division=0)), 4),
            "support": int(yt.sum()), "n_valid_samples": int(col.sum()),
            "confusion_matrix": confusion_matrix(yt, yp, labels=[0, 1]).tolist(),
        }
        f1s.append(f1)
    return {"macro_f1": round(float(np.mean(f1s)), 4) if f1s else None,
            "n_classes_evaluated": len(f1s), "per_class": per_class}


def print_pathology_report(metrics: Dict):
    print(f"\n{'Patología':<15}{'F1':>8}{'Precision':>12}{'Recall':>10}{'Support':>10}")
    print("-" * 55)
    for name in PATHOLOGY_NAMES:
        m = metrics["per_class"][name]
        if m["f1"] is None:
            print(f"{name:<15}{'n/a':>8}{'n/a':>12}{'n/a':>10}{'n/a':>10}   (sin etiquetas válidas)")
        else:
            print(f"{name:<15}{m['f1']:>8.3f}{m['precision']:>12.3f}{m['recall']:>10.3f}{m['support']:>10d}")
    print("-" * 55)
    f1m = metrics["macro_f1"]
    print(f"{'F1 macro':<15}{(f'{f1m:.3f}' if f1m is not None else 'n/a'):>8}"
          f"   ({metrics['n_classes_evaluated']} clases evaluadas)")
    print("\nMatrices de confusión ([[TN, FP], [FN, TP]]):")
    for name in PATHOLOGY_NAMES:
        print(f"  {name:<15}: {metrics['per_class'][name]['confusion_matrix']}")

<h2>Entrenamiento y evaluación</h2>

In [ ]:
METER_KEYS = ["loss", "mse", "pearson", "smooth", "pathology_bce"]


def train_one_epoch(model, loader, optimizer, criterion, device, epoch, epochs):
    model.train()
    meters, n = {k: 0.0 for k in METER_KEYS}, 0
    pbar = tqdm(loader, desc=f"Epoch {epoch}/{epochs} [train]", unit="batch", leave=False, dynamic_ncols=True)
    for x, y, y_patho, valid in pbar:
        x, y, y_patho, valid = x.to(device), y.to(device), y_patho.to(device), valid.to(device)
        optimizer.zero_grad()
        loss_dict = criterion(model(x), y, target_pathology=y_patho, valid_mask=valid)
        loss_dict["loss"].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        bs = x.shape[0]
        n += bs
        for k in meters:
            if k in loss_dict:
                meters[k] += loss_dict[k].item() * bs
        pbar.set_postfix(loss=f"{meters['loss'] / n:.4f}", mse=f"{meters['mse'] / n:.4f}",
                         pear=f"{meters['pearson'] / n:.3f}", lr=f"{optimizer.param_groups[0]['lr']:.1e}")
    pbar.close()
    return {k: v / max(n, 1) for k, v in meters.items()}


@torch.no_grad()
def evaluate(model, loader, criterion, device, desc="val"):
    model.eval()
    meters, n = {k: 0.0 for k in METER_KEYS}, 0
    preds, trues, p_logits, p_true, p_mask = [], [], [], [], []
    pbar = tqdm(loader, desc=f"[{desc}]", unit="batch", leave=False, dynamic_ncols=True)
    for x, y, y_patho, valid in pbar:
        x, y, y_patho, valid = x.to(device), y.to(device), y_patho.to(device), valid.to(device)
        pred = model(x)
        loss_dict = criterion(pred, y, target_pathology=y_patho, valid_mask=valid)
        preds.append(pred["ecg"]); trues.append(y)
        p_logits.append(pred["pathology_logits"]); p_true.append(y_patho); p_mask.append(valid)

        bs = x.shape[0]
        n += bs
        for k in meters:
            if k in loss_dict:
                meters[k] += loss_dict[k].item() * bs
        pbar.set_postfix(loss=f"{meters['loss'] / n:.4f}")
    pbar.close()

    out = {k: v / max(n, 1) for k, v in meters.items()}
    out.update(compute_metrics(torch.cat(preds), torch.cat(trues)))
    out["pathology_metrics"] = compute_pathology_metrics(
        torch.cat(p_logits), torch.cat(p_true), torch.cat(p_mask))
    return out

<h2>Diagnóstico de retraso PPG-ECG</h2>
Corre esta celda antes de entrenar. Lags mayores a ~0.1 s explican por qué un modelo causal no reconstruye el QRS.

In [ ]:
RUN_LAG_DIAGNOSTIC = True
if RUN_LAG_DIAGNOSTIC:
    _ds = BIDMCDataset(DEFAULT_BIDMC, split="train", T=500, stride=500)
    lags = ppg_ecg_lag(_ds)
    print(lags)
    print(f"lag mediano: {np.median(list(lags.values())):.3f} s")
    del _ds

<h2>Main</h2>

In [ ]:
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--bidmc_path", type=str, default=DEFAULT_BIDMC)
    parser.add_argument("--epochs", type=int, default=80)
    parser.add_argument("--batch_size", type=int, default=64)
    parser.add_argument("--T", type=int, default=500)
    parser.add_argument("--stride", type=int, default=250, help="stride de train y val")
    parser.add_argument("--test_stride", type=int, default=None, help="stride de test; por defecto T (sin solapamiento)")
    parser.add_argument("--hidden_dim", type=int, default=64)
    parser.add_argument("--n_layers", type=int, default=1)
    parser.add_argument("--down", type=int, default=2, help="submuestreo temporal del grafo (T/down pasos)")
    parser.add_argument("--dropout", type=float, default=0.15)
    parser.add_argument("--lr", type=float, default=5e-4)
    parser.add_argument("--patience", type=int, default=15)
    parser.add_argument("--output_dir", type=str, default="checkpointsModelo2")
    parser.add_argument("--seed", type=int, default=42)
    args, _ = parser.parse_known_args()

    args.output_dir = f"{args.output_dir}_h{args.hidden_dim}_l{args.n_layers}_d{args.down}_in3_v3"
    test_stride = args.test_stride or args.T

    set_seed(args.seed)
    os.makedirs(args.output_dir, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device} | salida: {args.output_dir}")

    train_ds = BIDMCDataset(args.bidmc_path, "train", args.T, args.stride, args.seed)
    val_ds = BIDMCDataset(args.bidmc_path, "val", args.T, args.stride, args.seed)
    test_ds = BIDMCDataset(args.bidmc_path, "test", args.T, test_stride, args.seed)

    n_workers = 0 if platform.system() == "Windows" else 2
    kw = dict(batch_size=args.batch_size, num_workers=n_workers, pin_memory=(device == "cuda"))
    train_loader = DataLoader(train_ds, shuffle=True, **kw)
    val_loader = DataLoader(val_ds, shuffle=False, **kw)
    test_loader = DataLoader(test_ds, shuffle=False, **kw)

    model = BioMEMOximetryToECG(
        build_default_heart_graph(device), input_dim=3, hidden_dim=args.hidden_dim, n_leads=1,
        n_layers=args.n_layers, dropout=args.dropout, down=args.down).to(device)
    print(f"Parámetros: {sum(p.numel() for p in model.parameters()):,}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
    criterion = ImprovedLoss(pos_weight=compute_pathology_pos_weight(train_ds).to(device))

    ckpt_path = os.path.join(args.output_dir, "best_model.pt")
    hist_path = os.path.join(args.output_dir, "history.json")
    start_epoch, best_val, bad_epochs, history = 1, float("inf"), 0, []

    if os.path.exists(ckpt_path):
        try:
            ckpt = torch.load(ckpt_path, map_location=device)
            model.load_state_dict(ckpt["model_state"])
            optimizer.load_state_dict(ckpt["optim_state"])
            if "sched_state" in ckpt:
                scheduler.load_state_dict(ckpt["sched_state"])
            start_epoch, best_val = ckpt["epoch"] + 1, ckpt["val_loss"]
            if os.path.exists(hist_path):
                with open(hist_path) as f:
                    history = json.load(f)
            print(f"Retomando desde epoch {start_epoch} | mejor val_loss: {best_val:.4f}")
        except RuntimeError as e:
            print(f"Checkpoint incompatible ({str(e)[:80]}...). Entrenando desde cero.")
    else:
        print("Sin checkpoint: entrenamiento desde cero")

    epoch_bar = tqdm(range(start_epoch, args.epochs + 1), desc="Entrenamiento", unit="epoch",
                     initial=start_epoch - 1, total=args.epochs, dynamic_ncols=True)
    for epoch in epoch_bar:
        t0 = time.time()
        tr = train_one_epoch(model, train_loader, optimizer, criterion, device, epoch, args.epochs)
        va = evaluate(model, val_loader, criterion, device, desc=f"Epoch {epoch}/{args.epochs} val")
        scheduler.step(va["loss"])

        history.append(to_native({"epoch": epoch, "train": tr, "val": va}))
        with open(hist_path, "w") as f:
            json.dump(history, f, indent=2)

        f1m = va["pathology_metrics"]["macro_f1"]
        epoch_bar.set_postfix(train=f"{tr['loss']:.3f}", val=f"{va['loss']:.3f}",
                              corr=f"{va['corr_mean']:.3f}", best=f"{min(best_val, va['loss']):.3f}",
                              lr=f"{optimizer.param_groups[0]['lr']:.1e}")
        tqdm.write(
            f"Epoch {epoch:03d} ({time.time() - t0:.0f}s) | Train Loss={tr['loss']:.4f} "
            f"MSE={tr['mse']:.4f} Pearson={tr['pearson']:.4f} | Val Loss={va['loss']:.4f} "
            f"RMSE={va['rmse_mean']:.4f} Corr={va['corr_mean']:.4f} PRD={va['prd_mean']:.2f}% "
            f"| Patho F1={'n/a' if f1m is None else f'{f1m:.3f}'}")

        if va["loss"] < best_val:
            best_val, bad_epochs = va["loss"], 0
            torch.save({"epoch": epoch, "model_state": model.state_dict(),
                        "optim_state": optimizer.state_dict(), "sched_state": scheduler.state_dict(),
                        "val_loss": va["loss"], "val_corr": va["corr_mean"], "args": vars(args)}, ckpt_path)
            tqdm.write(f"  Guardado nuevo mejor modelo (val_loss={best_val:.4f})")
        else:
            bad_epochs += 1
            if bad_epochs >= args.patience:
                tqdm.write(f"\nEarly stopping en epoch {epoch} (sin mejora por {args.patience} épocas)")
                break
    epoch_bar.close()

    print("\n--- Evaluación en test set ---")
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    test = evaluate(model, test_loader, criterion, device, desc="test")
    print(f"Checkpoint de epoch {ckpt['epoch']} | ventanas de test: {len(test_ds)} (stride={test_stride})")
    print(f"Test RMSE: {test['rmse_mean']:.4f} | Corr: {test['corr_mean']:.4f} | PRD: {test['prd_mean']:.2f}%")

    patho = test["pathology_metrics"]
    print("\n=== Clasificación de patologías (test) ===")
    print_pathology_report(patho)

    log_path = os.path.join(args.output_dir, "pathology_test_runs.json")
    runs = json.load(open(log_path)) if os.path.exists(log_path) else []
    current = {"timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
               "epoch": ckpt["epoch"], "macro_f1": patho["macro_f1"],
               "test_corr": round(test["corr_mean"], 4), "test_prd": round(test["prd_mean"], 2),
               "per_class_f1": {n: patho["per_class"][n]["f1"] for n in PATHOLOGY_NAMES}}

    print("\n" + "=" * 60)
    if runs and runs[-1]["epoch"] == current["epoch"]:
        print("Mismo checkpoint que la corrida anterior: no se registra de nuevo.")
        print("=" * 60)
        return
    if runs:
        last = runs[-1]
        print("COMPARACIÓN CON LA CORRIDA ANTERIOR")
        print(f"  Corr test : {last.get('test_corr')} -> {current['test_corr']}")
        print(f"  PRD test  : {last.get('test_prd')} -> {current['test_prd']}")
        if last["macro_f1"] is not None and current["macro_f1"] is not None:
            print(f"  F1 macro  : {last['macro_f1']:.4f} -> {current['macro_f1']:.4f} "
                  f"({current['macro_f1'] - last['macro_f1']:+.4f})")
        for n in PATHOLOGY_NAMES:
            a, b = last["per_class_f1"].get(n), current["per_class_f1"].get(n)
            if a is not None and b is not None:
                print(f"  F1 {n:<12}: {a:.4f} -> {b:.4f} ({b - a:+.4f})")
    else:
        print("Primera corrida registrada.")
    print("=" * 60)
    runs.append(current)
    with open(log_path, "w") as f:
        json.dump(runs, f, indent=2)


if __name__ == "__main__":
    main()